# Problema - Planejamento de Produção na Chocolaterie Floresta Viva

Pesquisa Operacional
**Aluno**: Matheus Sousa Marinho <br> 
**Matrícula**: 202206132    

**Problema**: A empresa **Chocolaterie Floresta Viva** produz três tipos de tabletes de chocolate premium:

- `x1`: tabletes 70% puro;
- `x2`: tabletes 70% com castanha;
- `x3`: tabletes 70% com cupuaçu.

O objetivo da empresa é determinar o plano mensal de produção que maximize o lucro, respeitando limitações de matéria-prima, capacidade produtiva e demanda.

A principal matéria-prima é o cacau amazônico fornecido pela cooperativa CAVF. A disponibilidade mensal de cacau, a capacidade mensal de produção e o custo de aquisição de cacau adicional variam conforme o cenário sazonal considerado. Os parâmetros do modelo são:

- `M`: disponibilidade de cacau (kg/mês);
- `C`: capacidade máxima de produção (unidades/mês);
- `c`: custo de compra extra de cacau (R$/kg).

Considere os seguintes cenários:

| Cenário | M (kg/mês) | C (unidades/mês) | c (R$/kg) |
|---|---:|---:|---:|
| A | 900 | 16000 | 55 |
| B | 400 | 10000 | 80 |
| C | 600 | 13500 | 62 |

As demandas máximas mensais são fixas para todos os cenários:

- produto 1: `D1 = 3000`;
- produto 2: `D2 = 4500`;
- produto 3: `D3 = 3500`.

A contribuição unitária para o lucro de cada produto é dada por:

- produto 1: R$ 18,50 por unidade;
- produto 2: R$ 24,80 por unidade;
- produto 3: R$ 21,60 por unidade.

O consumo de cacau por unidade produzida é:

- produto 1: 0,056 kg/unidade;
- produto 2: 0,044 kg/unidade;
- produto 3: 0,048 kg/unidade.

Admita ainda as seguintes variáveis auxiliares:

- `s+`: quantidade de cacau excedente devolvido (kg);
- `s-`: quantidade de cacau adicional adquirida (kg).

## Questṍes:

a) Formular o modelo de programação linear correspondente, definindo claramente as variáveis de decisão, a função objetivo e as restrições.

b) Resolver o modelo para cada um dos três cenários sazonais.

c) Para cada cenário, apresentar o plano ótimo de produção, a quantidade de cacau devolvida ou adquirida e o valor ótimo da função objetivo.

d) Comparar os resultados obtidos nos três cenários e discutir, de forma objetiva, os efeitos da disponibilidade de cacau, da capacidade produtiva e do custo de aquisição adicional sobre a decisão ótima.


# Solução do problema:
**a)** ### Variáveis de decisão: Neste problema temos três variáveis de decisão:

- `x1`: quantidade de tabletes 70% puro produzidos;
- `x2`: quantidade de tabletes 70% com castanha produzidos;
- `x3`: quantidade de tabletes 70% com cupuaçu produzidos.

Além disso, temos duas variáveis auxiliares:

- `s+`: quantidade de cacau excedente devolvido (kg);
- `s-`: quantidade de cacau adicional adquirida (kg).

### Função objetivo:
Neste problema, queremos determinar o plano mensal de produção que maximize o lucro. Sabemos que L(x)=R(x)-C(x), onde R(x) é a receita e C(x) é o custo.

Assim, a função objetivo é dada pela receita que remos menos o custo de produção de cada produto. O custo em particular é dado pelo custo do cacau extra * quantidade adquirida.

$$
\quad z = (18,5x_1 + 24,8x_2 + 21,6x_3) - c(s⁻)
$$

### Restrições (pensando nos demais cenários):
i) Demandas máximas fixas:
$$
x_1 <= 3000 \\
x_2 <= 4500 \\
x_3 <= 3500
$$ 

ii) Disponibilidade de cacau:
$$
0,056x_1 + 0,044x_2 + 0,048x_3 + s^+ - s⁻ <= M
$$

iii) Não-negatividade:
$$
x_1, x_2, x_3, s^+, s^- >= 0
$$










## Configurando o ambiente

Alinhado ao padrão utilizado em exercícios anteriores: `amplpy` + módulo `open` (solver CBC).

In [1]:
# 1. Instalação
%pip install -U -q amplpy
import sys, subprocess

subprocess.check_call([sys.executable, "-m", "amplpy.modules", "install", "--upgrade", "open"])

Note: you may need to restart the kernel to use updated packages.
$ /home/m9t/Documents/ufg/o-research/.venv/bin/python -m pip install -i https://pypi.ampl.com ampl_module_base ampl_module_open --upgrade
Looking in indexes: https://pypi.ampl.com
Imported ampl_module_base.
Imported ampl_module_base.
Imported ampl_module_open.


0

In [2]:
# 2. PATH do AMPL e importação
import os, sys, subprocess

caminho_mod = subprocess.check_output([sys.executable, "-m", "amplpy.modules", "path"]).decode().strip()
os.environ["PATH"] += os.pathsep + caminho_mod

from amplpy import AMPL

print("AMPL pronto (solver padrão configurado por modelo abaixo).")

AMPL pronto (solver padrão configurado por modelo abaixo).


## AMPL: Cenário A

In [ ]:
def cenarioA():
    """AMPL solver para o cenário A."""
    ampl = AMPL()
    ampl.eval("option solver cbc;")
    ampl.eval(
        """
        var x1 >= 0;
        var x2 >= 0;
        var x3 >= 0;
        var s_pl >= 0;
        var s_min >= 0;

        maximize lucro: 18.5*x1 + 24.8*x2 + 21.6*x3 - 55*s_min;

        subject to
        demanda_x1: x1 <= 3000;
        demanda_x2: x2 <= 4500;
        demanda_x3: x3 <= 3500;
        capacidade: x1 + x2 + x3 <= 16000;
        balanco_cacau: 0.056*x1 + 0.044*x2 + 0.048*x3 + s_pl - s_min = 900;
        """
    )
    ampl.solve()

    # Variaveis de interesse
    x1 = ampl.get_variable("x1").value()
    x2 = ampl.get_variable("x2").value()
    x3 = ampl.get_variable("x3").value()
    s_pl = ampl.get_variable("s_pl").value()
    s_min = ampl.get_variable("s_min").value()
    lucro = ampl.get_objective("lucro").value()

    # Indicadores uteis
    producao_total = x1 + x2 + x3
    cacau_usado = 0.056 * x1 + 0.044 * x2 + 0.048 * x3

    print("=== CENARIO A ===")
    print(f"x1 (70% puro): {x1:.2f}")
    print(f"x2 (70% castanha): {x2:.2f}")
    print(f"x3 (70% cupuacu): {x3:.2f}")
    print(f"s_pl (cacau devolvido, kg): {s_pl:.2f}")
    print(f"s_min (cacau comprado, kg): {s_min:.2f}")
    print(f"Producao total (unid): {producao_total:.2f}")
    print(f"Cacau usado (kg): {cacau_usado:.2f}")
    print(f"Lucro otimo: R$ {lucro:.2f}")

    return ampl


ampl_prob = cenarioA()

cbc 2.10.12:cbc 2.10.12: optimal solution; objective 242700
0 simplex iterations
=== CENARIO A ===
x1 (70% puro): 3000.00
x2 (70% castanha): 4500.00
x3 (70% cupuacu): 3500.00
s_pl (cacau devolvido, kg): 366.00
s_min (cacau comprado, kg): 0.00
Producao total (unid): 11000.00
Cacau usado (kg): 534.00
Lucro otimo: R$ 242700.00


## AMPL: Cenário B

In [ ]:
def cenarioB():
    """AMPL solver para o cenário B. (M = 400, C = 10000, c = 80)"""
    ampl = AMPL()
    ampl.eval("option solver cbc;")
    ampl.eval(
        """
        var x1 >= 0;
        var x2 >= 0;
        var x3 >= 0;
        var s_pl >= 0;
        var s_min >= 0;

        maximize lucro: 18.5*x1 + 24.8*x2 + 21.6*x3 - 80*s_min;

        subject to
        demanda_x1: x1 <= 3000;
        demanda_x2: x2 <= 4500;
        demanda_x3: x3 <= 3500;
        capacidade: x1 + x2 + x3 <= 10000;
        balanco_cacau: 0.056*x1 + 0.044*x2 + 0.048*x3 + s_pl - s_min = 400;
        """
    )
    ampl.solve()

    # Variaveis de interesse
    x1 = ampl.get_variable("x1").value()
    x2 = ampl.get_variable("x2").value()
    x3 = ampl.get_variable("x3").value()
    s_pl = ampl.get_variable("s_pl").value()
    s_min = ampl.get_variable("s_min").value()
    lucro = ampl.get_objective("lucro").value()

    # Indicadores uteis
    producao_total = x1 + x2 + x3
    cacau_usado = 0.056 * x1 + 0.044 * x2 + 0.048 * x3

    print("=== CENARIO B===")
    print(f"x1 (70% puro): {x1:.2f}")
    print(f"x2 (70% castanha): {x2:.2f}")
    print(f"x3 (70% cupuacu): {x3:.2f}")
    print(f"s_pl (cacau devolvido, kg): {s_pl:.2f}")
    print(f"s_min (cacau comprado, kg): {s_min:.2f}")
    print(f"Producao total (unid): {producao_total:.2f}")
    print(f"Cacau usado (kg): {cacau_usado:.2f}")
    print(f"Lucro otimo: R$ {lucro:.2f}")

    return ampl


ampl_prob = cenarioB()

cbc 2.10.12:cbc 2.10.12: optimal solution; objective 217960
0 simplex iterations
=== CENARIO B===
x1 (70% puro): 2000.00
x2 (70% castanha): 4500.00
x3 (70% cupuacu): 3500.00
s_pl (cacau devolvido, kg): 0.00
s_min (cacau comprado, kg): 78.00
Producao total (unid): 10000.00
Cacau usado (kg): 478.00
Lucro otimo: R$ 217960.00


## AMPL: Cenário C

In [14]:
def cenarioC():
    """AMPL solver para o cenário C. (M = 600, C = 13500, c = 62)"""
    ampl = AMPL()
    ampl.eval("option solver cbc;")
    ampl.eval(
        """
        var x1 >= 0;
        var x2 >= 0;
        var x3 >= 0;
        var s_pl >= 0;
        var s_min >= 0;

        maximize lucro: 18.5*x1 + 24.8*x2 + 21.6*x3 - 62*s_min;

        subject to
        demanda_x1: x1 <= 3000;
        demanda_x2: x2 <= 4500;
        demanda_x3: x3 <= 3500;
        capacidade: x1 + x2 + x3 <= 13500;
        balanco_cacau: 0.056*x1 + 0.044*x2 + 0.048*x3 + s_pl - s_min = 600;
        """
    )
    ampl.solve()

    # Variaveis de interesse
    x1 = ampl.get_variable("x1").value()
    x2 = ampl.get_variable("x2").value()
    x3 = ampl.get_variable("x3").value()
    s_pl = ampl.get_variable("s_pl").value()
    s_min = ampl.get_variable("s_min").value()
    lucro = ampl.get_objective("lucro").value()

    # Indicadores uteis
    producao_total = x1 + x2 + x3
    cacau_usado = 0.056 * x1 + 0.044 * x2 + 0.048 * x3

    print("=== CENARIO C ===")
    print(f"x1 (70% puro): {x1:.2f}")
    print(f"x2 (70% castanha): {x2:.2f}")
    print(f"x3 (70% cupuacu): {x3:.2f}")
    print(f"s_pl (cacau devolvido, kg): {s_pl:.2f}")
    print(f"s_min (cacau comprado, kg): {s_min:.2f}")
    print(f"Producao total (unid): {producao_total:.2f}")
    print(f"Cacau usado (kg): {cacau_usado:.2f}")
    print(f"Lucro otimo: R$ {lucro:.2f}")

    return ampl


ampl_prob = cenarioC()

cbc 2.10.12:cbc 2.10.12: optimal solution; objective 242700
0 simplex iterations
=== CENARIO C ===
x1 (70% puro): 3000.00
x2 (70% castanha): 4500.00
x3 (70% cupuacu): 3500.00
s_pl (cacau devolvido, kg): 66.00
s_min (cacau comprado, kg): 0.00
Producao total (unid): 11000.00
Cacau usado (kg): 534.00
Lucro otimo: R$ 242700.00


## **Comparação final:**

## Tabela-resumo dos resultados finais

| Cenário | x1 (70% puro) | x2 (70% castanha) | x3 (70% cupuaçu) | Produção total (unid) | Cacau usado (kg) | s_pl (devolvido, kg) | s_min (comprado, kg) | Lucro ótimo (R$) |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| A | 3000 | 4500 | 3500 | 11000 | 534 | 366 | 0 | 242700 |
| B | 2000 | 4500 | 3500 | 10000 | 478 | 0 | 78 | 217960 |
| C | 3000 | 4500 | 3500 | 11000 | 534 | 66 | 0 | 242700 |

Com base nos resultados obtidos, 


---

**Referências:** Goldbarg & Luna (2006); [AMPL](https://ampl.com); [HiGHS](https://highs.dev) / CBC via `amplpy` open module.